# 追加実験<font color="red">【没】</font>
* 卒業研究では使用していない（DOI付与の判断や，十分な数のデータを集めるのに苦戦）
* ~~実際の参考文献文字列で実験~~
* ~~人工知能学会論文誌36巻(2021年)に採択された論文に含まれる和文参考文献文字列194件を使用~~
* ~~Jastageによりリンクが貼られていた(すなわちDOIが付与)ものは<font color="red">113件</font>，貼られていなかったものは<font color="green">81件</font>~~

## <font color="red">要実行</font>

In [1]:
# モジュールの読み込み
from pathlib import Path
import sys
import pysolr

TOOLS_DIR = Path.cwd().resolve().parent / "8プロジェクト関連" / "8-3共有プログラム"
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

import wakati
import generate_query
import matplotlib.pyplot as plt

# 接続文字列
solr_url = "http://localhost:8983/solr/jalc"

# pysolrのクライアントの初期化
solr = pysolr.Solr(solr_url,timeout=10)

# 参考文献文字列の読み込み
positive_test_set = []
negative_test_set = []

# 参考文献文字列(正例)の読み込み
with open('../5データ/5-2評価実験用データ/5-2-1参考文献文字列（実在）/positive_reference_real.txt','r') as f:
    lines = f.readlines()
# 文末の改行除去
for line in lines:
    positive_test_set.append(line.strip())

# 参考文献文字列(負例)の読み込み
with open('../5データ/5-2評価実験用データ/5-2-2参考文献文字列（架空）/negative_reference_real.txt','r') as f:
    lines = f.readlines()
# 文末の改行除去
for line in lines:
    negative_test_set.append(line.strip())

# doiの読み込み
doi_list = []
with open('../5データ/5-2評価実験用データ/5-2-1参考文献文字列（実在）/real_reference_doi_list.txt','r') as f:
    lines = f.readlines()

# 文末の改行除去
doi_list = []
for line in lines:
    doi_list.append(line.strip())        

## BM25

### 参考文献文字列（実在）

In [3]:
BM25_threshold = 38.92128   # BM25の閾値
# 検索オプション
search_options = {
    'q.op':'OR',
    'fl':'doi,first_author,creator,title,journal,issued,volume_issue,page_range,score',
    'df':'jalcdata',
    'rows':1
}

# 変数・リストの初期化
count = 0
true_positive = 0
false_negative = 0
experimental_results = []   # 実験結果を補完するリスト

for reference in positive_test_set:
    # エスケープ処理
    escape_query = generate_query.escape_solr_query(reference)
    # solrによる検索
    results = solr.search(escape_query,**search_options)
    results = list(results)
    candidate_document_doi = "" # 候補文献のdoi
    # 閾値との大小比較
    if results[0]["score"] >= BM25_threshold:
        candidate_document_doi = results[0]["doi"]
    else:
        candidate_document_doi = "No registration"   
    # doiの照合
    if candidate_document_doi == doi_list[count]:
        experimental_results.append([reference,doi_list[count],candidate_document_doi,"match"])
        true_positive += 1
    else:
        if results[0]["doi"] == doi_list[count]:
            experimental_results.append([reference,doi_list[count],candidate_document_doi,"mismatch","閾値を超えなかった"])
        else:
            experimental_results.append([reference,doi_list[count],candidate_document_doi,"mismatch","誤った論文を同定"])    
        false_negative += 1
    count += 1    

# 標準出力
print(f"True positive : {true_positive}")
print(f"False negative : {false_negative}")
print("="*20)
for val in experimental_results:
    print(*val)            

True positive : 110
False negative : 3
梶原健吾，鳥海不二夫，稲葉通将：人狼における強化学習を用いたエージェントの設計，2015年度人工知能学会全国大会（第29回）論文集，Vol. 29, pp. 1–3 (2015). 10.11517/pjsai.JSAI2015.0_1F22 10.11517/pjsai.JSAI2015.0_1F22 match
片上大輔, 鳥海不二夫, 大澤博隆, 稲葉通将, 篠田孝祐, 松原仁, 人狼知能プロジェクト, 人工知能学会論文誌，Vol.30 No.1，pp.65-73, 2015. 10.11517/jjsai.30.1_65 10.11517/jjsai.30.1_65 match
赤坂文弥, 木村篤信: リビングラボの方法論的特徴の分析,日本デザイン学会第64回春季研究発表大会概要集, B1-05, pp. 22-23 (2017) 10.11247/jssd.64.0_22 10.11247/jssd.64.0_22 match
堀田竜士，三井実，伊藤孝行，白松俊，藤田桂英，福田直樹: 研究者と市民の共創を生み出す研究会の提案, 人工知能, 第34巻, 4号, pp. D-I92_1-8 (2019) 10.1527/tjsai.D-I92 10.1527/tjsai.D-I92 match
堀田竜士, 仙石晃久, 伊藤孝行: 多地域移動型の共創を支援するソーシャルメディアシステムの試作と評価, 人工知能, 第34巻, 2号, pp. F-I57_1-8 (2019) 10.1527/tjsai.F-I57 10.1527/tjsai.F-I57 match
一ノ瀬修吾, 白松俊, 大森友子: Kinectを用いた鍬動作の初心者と熟練者の比較分析手法の試作, 第31回人工知能学会全国大会, 2E4-OS-36b-3in1 (2017) 10.11517/pjsai.JSAI2017.0_2E4OS36b3 10.11517/pjsai.JSAI2017.0_2E4OS36b3 match
一ノ瀬修吾, 白松俊, 大森友子: Kinectを用いた鍬動作分析研究における市民共創知研究会を通じた今後の展望, 第2回市民共創知研究会 (2017) 10.11517/jsaisigtwo.2

### 参考文献文字列（架空）

In [4]:
BM25_threshold = 38.92128   # BM25の閾値
# 検索オプション
search_options = {
    'q.op':'OR',
    'fl':'doi,first_author,creator,title,journal,issued,volume_issue,page_range,score',
    'df':'jalcdata',
    'rows':1
}

# 変数・リストの初期化
count = 0
true_negative = 0
false_positive = 0
experimental_results = []   # 実験結果を補完するリスト

for reference in negative_test_set:
    # エスケープ処理
    escape_query = generate_query.escape_solr_query(reference)
    # solrによる検索
    results = solr.search(escape_query,**search_options)
    results = list(results)
    candidate_document_doi = "" # 候補文献のdoi
    # 閾値との大小比較
    if results[0]["score"] >= BM25_threshold:
        candidate_document_doi = results[0]["doi"]
    else:
        candidate_document_doi = "No registration"    
    # doiの照合
    if candidate_document_doi == "No registration":
        experimental_results.append([reference,"No registration",candidate_document_doi,"match"])
        true_negative += 1
    else:
        experimental_results.append([reference,"No registration",candidate_document_doi,"mismatch"])
        false_positive += 1
    count += 1    

# 標準出力
print(f"True negative : {true_negative}")
print(f"False positive : {false_positive}")
print("="*20)
for val in experimental_results:
    print(*val)            

True negative : 45
False positive : 36
北島瑛貴, 上野史, 村田暁紀, 髙玉圭樹：好奇心を持つエージェントによる多様性のある情報伝搬シミュレーションモデルの提案, HAI シンポジウム2018 (2018) No registration No registration match
畢暁恒，田中哲朗: 対話のない人狼ゲームの戦略, ゲームプログラミングワークショップ2015論文集, pp.25–30, 2015. No registration No registration match
飯田弘之, 松原仁: ゲーム情報学の動向, 情報処理, vol.44, No.9, pp.895-899, 2003. No registration No registration match
稲葉通将，鳥海不二夫，石井健一郎：語の共起情報を用いた対話における盛り上がりの自動判定，信学論（D），Vol. 94, No. 1, pp. 59-67, 2011. No registration 10.11517/pjsai.JSAI2011.0_3C2OS196 mismatch
近藤まなみ, 長谷川拓, 森直樹, 松本啓之亮, LSTM を用いた人狼予測と人狼ゲーム分析. 2018 年度人工知能学会全国大会（第32回）論文集, 2018 No registration 10.11517/pjsai.JSAI2018.0_1N103 mismatch
小柴 等, 相原健郎, 小田朋宏, 星孝哲, 松原伸人, 森純一郎, 武田英明ほか :説得性に基づく情報推薦手法の提案: 送り手の属性に着目したモデルと検証, 情報処理学会論文誌, Vol. 51, No. 8, pp. 1452–1468, 2010. No registration No registration match
小谷善行: 第3回将棋電王戦を振り返って：3．コンピュータ将棋の棋力の客観的分析～人間のトップに到達したか？～, 情報処理, Vol.55, No.8, pp.851–852, 2014. No registration No registration match
大川貴聖，吉仲亮，篠原歩: 深層学習を用いて役職推定を行う人狼知能エージェントの開発, The 22nd G

## RC(Reference Coverage)

### 参考文献文字列（実在）

In [5]:
# リスト型データの各要素をトークン化して結合
def list_to_token(lis):
    tokens = []
    for val in lis:
        tokens += wakati.get_token(val)
    return tokens

# sim_r 算出用の候補文献のトークン集合を得る
def get_sim_r_token(document):
    tokens = []
    # トークン化するフィールド一覧
    field_names = ["creator","title","journal","issued","volume_issue","page_range"]
    # 各フィールドをトークン化して結合
    for field_name in field_names:
        tokens += list(set(list_to_token(document[field_name])))    
    return tokens    

# sim_r の算出
def calc_sim_r(reference_token,sim_r_token):
    # sim_r スコア
    sim_r_score = 0
    # 両方のトークン集合の共通要素の数
    common_token_num = 0

    for token in reference_token:
        if str(token) in sim_r_token:
            common_token_num += 1
        """    
        else:
            print(f"含まれなかったトークン：{token}")
        """        

    sim_r_score = common_token_num / len(reference_token)

    return sim_r_score            

# sim_rの閾値
sim_r_threshold = 0.9375

# 検索オプション
search_options = {
    'q.op':'OR',
    'fl':'doi,first_author,creator,title,journal,issued,volume_issue,page_range,score',
    'df':'jalcdata',
    'rows':10
}

# 変数・リストの初期化
count = 0
true_positive = 0
false_negative = 0
experimental_results = []   # 実験結果を補完するリスト

for reference in positive_test_set:
    # エスケープ処理
    escape_query = generate_query.escape_solr_query(reference)
    # 参考文献文字列のトークン化
    reference_token = list(set(wakati.get_token(reference)))
    # solrによる全文検索
    results = solr.search(escape_query,**search_options)
    # リランキング処理
    max_sim_r_score = -1
    candidate_document = "" # 最大スコアの候補文献
    for result in results:
        # 候補文献のトークン化
        sim_r_token = get_sim_r_token(result)
        # sim_rの算出
        sim_r_score = calc_sim_r(reference_token,sim_r_token) 
        if max_sim_r_score < sim_r_score:
            candidate_document = result
            max_sim_r_score = sim_r_score
    # 閾値との大小比較
    candidate_document_doi = ""
    if max_sim_r_score >= sim_r_threshold:
        candidate_document_doi = candidate_document["doi"]
    else:
        candidate_document_doi = "No registration"
    # doiの照合
    if candidate_document_doi == doi_list[count]:
        experimental_results.append([reference,doi_list[count],candidate_document_doi,"match"])
        true_positive += 1
    else:
        if candidate_document["doi"] == doi_list[count]:
            experimental_results.append([reference,doi_list[count],candidate_document_doi,max_sim_r_score,"mismatch","閾値を超えなかった"])
        else:
            experimental_results.append([reference,doi_list[count],candidate_document_doi,"mismatch","誤った論文を同定"])    
        false_negative += 1
    count += 1

# 標準出力
print(f"True positive : {true_positive}")
print(f"False negative : {false_negative}")
print("="*20)
for val in experimental_results:
    print(*val)                    


True positive : 89
False negative : 24
梶原健吾，鳥海不二夫，稲葉通将：人狼における強化学習を用いたエージェントの設計，2015年度人工知能学会全国大会（第29回）論文集，Vol. 29, pp. 1–3 (2015). 10.11517/pjsai.JSAI2015.0_1F22 No registration 0.9 mismatch 閾値を超えなかった
片上大輔, 鳥海不二夫, 大澤博隆, 稲葉通将, 篠田孝祐, 松原仁, 人狼知能プロジェクト, 人工知能学会論文誌，Vol.30 No.1，pp.65-73, 2015. 10.11517/jjsai.30.1_65 No registration 0.8717948717948718 mismatch 閾値を超えなかった
赤坂文弥, 木村篤信: リビングラボの方法論的特徴の分析,日本デザイン学会第64回春季研究発表大会概要集, B1-05, pp. 22-23 (2017) 10.11247/jssd.64.0_22 No registration 0.8297872340425532 mismatch 閾値を超えなかった
堀田竜士，三井実，伊藤孝行，白松俊，藤田桂英，福田直樹: 研究者と市民の共創を生み出す研究会の提案, 人工知能, 第34巻, 4号, pp. D-I92_1-8 (2019) 10.1527/tjsai.D-I92 10.1527/tjsai.D-I92 match
堀田竜士, 仙石晃久, 伊藤孝行: 多地域移動型の共創を支援するソーシャルメディアシステムの試作と評価, 人工知能, 第34巻, 2号, pp. F-I57_1-8 (2019) 10.1527/tjsai.F-I57 10.1527/tjsai.F-I57 match
一ノ瀬修吾, 白松俊, 大森友子: Kinectを用いた鍬動作の初心者と熟練者の比較分析手法の試作, 第31回人工知能学会全国大会, 2E4-OS-36b-3in1 (2017) 10.11517/pjsai.JSAI2017.0_2E4OS36b3 10.11517/pjsai.JSAI2017.0_2E4OS36b3 match
一ノ瀬修吾, 白松俊, 大森友子: Kinectを用いた鍬動作分析研究における市民共創知研

### 参考文献文字列（架空）

In [6]:
# リスト型データの各要素をトークン化して結合
def list_to_token(lis):
    tokens = []
    for val in lis:
        tokens += wakati.get_token(val)
    return tokens

# sim_r 算出用の候補文献のトークン集合を得る
def get_sim_r_token(document):
    tokens = []
    # トークン化するフィールド一覧
    field_names = ["creator","title","journal","issued","volume_issue","page_range"]
    # 各フィールドをトークン化して結合
    for field_name in field_names:
        tokens += list(set(list_to_token(document[field_name])))    
    return tokens    

# sim_r の算出
def calc_sim_r(reference_token,sim_r_token):
    # sim_r スコア
    sim_r_score = 0
    # 両方のトークン集合の共通要素の数
    common_token_num = 0

    for token in reference_token:
        if str(token) in sim_r_token:
            common_token_num += 1
        """    
        else:
            print(f"含まれなかったトークン：{token}")
        """        

    sim_r_score = common_token_num / len(reference_token)

    return sim_r_score            

# sim_rの閾値
sim_r_threshold = 0.9375

# 検索オプション
search_options = {
    'q.op':'OR',
    'fl':'doi,first_author,creator,title,journal,issued,volume_issue,page_range,score',
    'df':'jalcdata',
    'rows':10
}

# 変数・リストの初期化
count = 0
true_negative = 0
false_positive = 0
experimental_results = []   # 実験結果を補完するリスト

for reference in negative_test_set:
    # エスケープ処理
    escape_query = generate_query.escape_solr_query(reference)
    # 参考文献文字列のトークン化
    reference_token = list(set(wakati.get_token(reference)))
    # solrによる全文検索
    results = solr.search(escape_query,**search_options)
    # リランキング処理
    max_sim_r_score = -1
    candidate_document = "" # 最大スコアの候補文献
    for result in results:
        # 候補文献のトークン化
        sim_r_token = get_sim_r_token(result)
        # sim_rの算出
        sim_r_score = calc_sim_r(reference_token,sim_r_token) 
        if max_sim_r_score < sim_r_score:
            candidate_document = result
            max_sim_r_score = sim_r_score
    # 閾値との大小比較
    candidate_document_doi = ""
    if max_sim_r_score >= sim_r_threshold:
        candidate_document_doi = candidate_document["doi"]
    else:
        candidate_document_doi = "No registration"
    # doiの照合
    if candidate_document_doi == "No registration":
        experimental_results.append([reference,"No registration",candidate_document_doi,"match"])
        true_negative += 1
    else:
        experimental_results.append([reference,"No registration",candidate_document_doi,"mismatch"])
        false_positive += 1
    count += 1

# 標準出力
print(f"True negative : {true_negative}")
print(f"False positive : {false_positive}")
print("="*20)
for val in experimental_results:
    print(*val)                    


True negative : 70
False positive : 11
北島瑛貴, 上野史, 村田暁紀, 髙玉圭樹：好奇心を持つエージェントによる多様性のある情報伝搬シミュレーションモデルの提案, HAI シンポジウム2018 (2018) No registration No registration match
畢暁恒，田中哲朗: 対話のない人狼ゲームの戦略, ゲームプログラミングワークショップ2015論文集, pp.25–30, 2015. No registration No registration match
飯田弘之, 松原仁: ゲーム情報学の動向, 情報処理, vol.44, No.9, pp.895-899, 2003. No registration No registration match
稲葉通将，鳥海不二夫，石井健一郎：語の共起情報を用いた対話における盛り上がりの自動判定，信学論（D），Vol. 94, No. 1, pp. 59-67, 2011. No registration No registration match
近藤まなみ, 長谷川拓, 森直樹, 松本啓之亮, LSTM を用いた人狼予測と人狼ゲーム分析. 2018 年度人工知能学会全国大会（第32回）論文集, 2018 No registration No registration match
小柴 等, 相原健郎, 小田朋宏, 星孝哲, 松原伸人, 森純一郎, 武田英明ほか :説得性に基づく情報推薦手法の提案: 送り手の属性に着目したモデルと検証, 情報処理学会論文誌, Vol. 51, No. 8, pp. 1452–1468, 2010. No registration No registration match
小谷善行: 第3回将棋電王戦を振り返って：3．コンピュータ将棋の棋力の客観的分析～人間のトップに到達したか？～, 情報処理, Vol.55, No.8, pp.851–852, 2014. No registration No registration match
大川貴聖，吉仲亮，篠原歩: 深層学習を用いて役職推定を行う人狼知能エージェントの開発, The 22nd Game Programming Workshop 2017, 2017. No r

## CC(Candidate Coverage)

### 参考文献文字列（実在）

In [7]:
# リスト型データの各要素のトークン化
def list_to_token(lis):
    tokens = []
    for val in lis:
        tokens.append(wakati.get_token(val))
    return tokens

# sim_p 算出用のトークン集合を得る
def get_sim_p_token(document):
    sim_p_token = []
    # トークン化するフィールド名一覧
    field_names = ["first_author","title","journal","issued","volume_issue","page_range"]
    # 各フィールドをトークン化
    for field in field_names:
        sim_p_token.append(list_to_token(document[field]))
    return sim_p_token

# sim_p 算出関数
def calc_sim_p(reference_token,sim_p_token):
    sim_p_score = 0
    for i in range(len(sim_p_token)):
        max_score = 0
        for j in range(len(sim_p_token[i])):
            common_token_num = 0
            for token in sim_p_token[i][j]:
                if token in reference_token:
                    common_token_num += 1
            tmp = common_token_num / len(sim_p_token[i][j])
            if tmp > max_score:
                max_score = tmp
        sim_p_score += max_score
    return sim_p_score / 6                   

# 空のリストを削除
def remove_empty_lists(data):
    if isinstance(data, list):
        # リストであれば、各要素に対して再帰的に処理
        return [remove_empty_lists(item) for item in data if remove_empty_lists(item)]
    else:
        # リストでなければ、そのまま返す
        return data

# sim_pの閾値
sim_p_threshold = 0.8730

# 検索オプション
search_options = {
    'q.op':'OR',
    'fl':'doi,first_author,creator,title,journal,issued,volume_issue,page_range,score',
    'df':'jalcdata',
    'rows':10
}

# 変数・リストの初期化
count = 0
true_positive = 0
false_negative = 0
experimental_results = []   # 実験結果を補完するリスト

for reference in positive_test_set:
    # エスケープ処理
    escape_query = generate_query.escape_solr_query(reference)
    # 参考文献文字列のトークン化
    reference_token = list(set(wakati.get_token(reference)))
    # solrによる全文検索
    results = solr.search(escape_query,**search_options)
    # リランキング処理
    max_sim_p_score = -1
    candidate_document = "" # 最大スコアの候補文献
    for result in results:
        # 候補文献のトークン化
        sim_p_token = get_sim_p_token(result)
        # 空のリストを除去
        sim_p_token = remove_empty_lists(sim_p_token)
        # sim_p の算出
        sim_p_score = calc_sim_p(reference_token,sim_p_token)
        if max_sim_p_score < sim_p_score:
            candidate_document = result
            max_sim_p_score = sim_p_score
    # 閾値との大小比較
    candidate_document_doi = ""
    if max_sim_p_score >= sim_p_threshold:
        candidate_document_doi = candidate_document["doi"]
    else:
        candidate_document_doi = "No registration"
    # doiの照合
    if candidate_document_doi == doi_list[count]:
        experimental_results.append([reference,doi_list[count],candidate_document_doi,max_sim_p_score,"match"])
        true_positive += 1
    else:
        if candidate_document["doi"] == doi_list[count]:
            experimental_results.append([reference,doi_list[count],candidate_document_doi,max_sim_p_score,"mismatch","閾値を超えなかった"])
        else:
            experimental_results.append([reference,doi_list[count],candidate_document_doi,"mismatch","誤った論文を同定"])    
        false_negative += 1
    count += 1                              

# 標準出力
print(f"True positive : {true_positive}")
print(f"False negative : {false_negative}")
print("="*20)
for val in experimental_results:
    print(*val) 

True positive : 90
False negative : 23
梶原健吾，鳥海不二夫，稲葉通将：人狼における強化学習を用いたエージェントの設計，2015年度人工知能学会全国大会（第29回）論文集，Vol. 29, pp. 1–3 (2015). 10.11517/pjsai.JSAI2015.0_1F22 No registration 0.7694444444444444 mismatch 閾値を超えなかった
片上大輔, 鳥海不二夫, 大澤博隆, 稲葉通将, 篠田孝祐, 松原仁, 人狼知能プロジェクト, 人工知能学会論文誌，Vol.30 No.1，pp.65-73, 2015. 10.11517/jjsai.30.1_65 10.11517/jjsai.30.1_65 0.9015151515151515 match
赤坂文弥, 木村篤信: リビングラボの方法論的特徴の分析,日本デザイン学会第64回春季研究発表大会概要集, B1-05, pp. 22-23 (2017) 10.11247/jssd.64.0_22 10.11247/jssd.64.0_22 0.9895833333333334 match
堀田竜士，三井実，伊藤孝行，白松俊，藤田桂英，福田直樹: 研究者と市民の共創を生み出す研究会の提案, 人工知能, 第34巻, 4号, pp. D-I92_1-8 (2019) 10.1527/tjsai.D-I92 10.1527/tjsai.D-I92 0.8958333333333334 match
堀田竜士, 仙石晃久, 伊藤孝行: 多地域移動型の共創を支援するソーシャルメディアシステムの試作と評価, 人工知能, 第34巻, 2号, pp. F-I57_1-8 (2019) 10.1527/tjsai.F-I57 10.1527/tjsai.F-I57 0.8958333333333334 match
一ノ瀬修吾, 白松俊, 大森友子: Kinectを用いた鍬動作の初心者と熟練者の比較分析手法の試作, 第31回人工知能学会全国大会, 2E4-OS-36b-3in1 (2017) 10.11517/pjsai.JSAI2017.0_2E4OS36b3 10.11517/pjsai.JSAI2017.0_2E4OS36b3 0.875 match

### 参考文献文字列（架空）

In [9]:
# リスト型データの各要素のトークン化
def list_to_token(lis):
    tokens = []
    for val in lis:
        tokens.append(wakati.get_token(val))
    return tokens

# sim_p 算出用のトークン集合を得る
def get_sim_p_token(document):
    sim_p_token = []
    # トークン化するフィールド名一覧
    field_names = ["first_author","title","journal","issued","volume_issue","page_range"]
    # 各フィールドをトークン化
    for field in field_names:
        sim_p_token.append(list_to_token(document[field]))
    return sim_p_token

# sim_p 算出関数
def calc_sim_p(reference_token,sim_p_token):
    sim_p_score = 0
    for i in range(len(sim_p_token)):
        max_score = 0
        for j in range(len(sim_p_token[i])):
            common_token_num = 0
            for token in sim_p_token[i][j]:
                if token in reference_token:
                    common_token_num += 1
            tmp = common_token_num / len(sim_p_token[i][j])
            if tmp > max_score:
                max_score = tmp
        sim_p_score += max_score
    return sim_p_score / 6                   

# 空のリストを削除
def remove_empty_lists(data):
    if isinstance(data, list):
        # リストであれば、各要素に対して再帰的に処理
        return [remove_empty_lists(item) for item in data if remove_empty_lists(item)]
    else:
        # リストでなければ、そのまま返す
        return data

# sim_pの閾値
sim_p_threshold = 0.8730

# 検索オプション
search_options = {
    'q.op':'OR',
    'fl':'doi,first_author,creator,title,journal,issued,volume_issue,page_range,score',
    'df':'jalcdata',
    'rows':10
}

# 変数・リストの初期化
count = 0
true_negative = 0
false_positive = 0
experimental_results = []   # 実験結果を補完するリスト

for reference in negative_test_set:
    # エスケープ処理
    escape_query = generate_query.escape_solr_query(reference)
    # 参考文献文字列のトークン化
    reference_token = list(set(wakati.get_token(reference)))
    # solrによる全文検索
    results = solr.search(escape_query,**search_options)
    # リランキング処理
    max_sim_p_score = -1
    candidate_document = "" # 最大スコアの候補文献
    for result in results:
        # 候補文献のトークン化
        sim_p_token = get_sim_p_token(result)
        # 空のリストを除去
        sim_p_token = remove_empty_lists(sim_p_token)
        # sim_p の算出
        sim_p_score = calc_sim_p(reference_token,sim_p_token)
        if max_sim_p_score < sim_p_score:
            candidate_document = result
            max_sim_p_score = sim_p_score
    # 閾値との大小比較
    candidate_document_doi = ""
    if max_sim_p_score >= sim_p_threshold:
        candidate_document_doi = candidate_document["doi"]
    else:
        candidate_document_doi = "No registration"
    # doiの照合
    if candidate_document_doi == "No registration":
        experimental_results.append([reference,"No registration",candidate_document_doi,"match",max_sim_p_score])
        true_negative += 1
    else:
        experimental_results.append([reference,"No registration",candidate_document_doi,"mismatch",max_sim_p_score])
        false_positive += 1
    count += 1

# 標準出力
print(f"True negative : {true_negative}")
print(f"False positive : {false_positive}")
print("="*20)
for val in experimental_results:
    print(*val)             
            

True negative : 69
False positive : 12
北島瑛貴, 上野史, 村田暁紀, 髙玉圭樹：好奇心を持つエージェントによる多様性のある情報伝搬シミュレーションモデルの提案, HAI シンポジウム2018 (2018) No registration No registration match 0.22522522522522523
畢暁恒，田中哲朗: 対話のない人狼ゲームの戦略, ゲームプログラミングワークショップ2015論文集, pp.25–30, 2015. No registration No registration match 0.41269841269841273
飯田弘之, 松原仁: ゲーム情報学の動向, 情報処理, vol.44, No.9, pp.895-899, 2003. No registration No registration match 0.4166666666666667
稲葉通将，鳥海不二夫，石井健一郎：語の共起情報を用いた対話における盛り上がりの自動判定，信学論（D），Vol. 94, No. 1, pp. 59-67, 2011. No registration No registration match 0.44298245614035087
近藤まなみ, 長谷川拓, 森直樹, 松本啓之亮, LSTM を用いた人狼予測と人狼ゲーム分析. 2018 年度人工知能学会全国大会（第32回）論文集, 2018 No registration No registration match 0.736111111111111
小柴 等, 相原健郎, 小田朋宏, 星孝哲, 松原伸人, 森純一郎, 武田英明ほか :説得性に基づく情報推薦手法の提案: 送り手の属性に着目したモデルと検証, 情報処理学会論文誌, Vol. 51, No. 8, pp. 1452–1468, 2010. No registration No registration match 0.3666666666666667
小谷善行: 第3回将棋電王戦を振り返って：3．コンピュータ将棋の棋力の客観的分析～人間のトップに到達したか？～, 情報処理, Vol.55, No.8, pp.851–852, 2014. No registration N